# Загрузка данных и формирование target

Сначала явно проверяем дубликаты файлов через MD5-хэши, затем импортируем только уникальные CSV с корректным приведением дат и времени.

In [ ]:
from pathlib import Path
import hashlib

import pandas as pd

SRC_DIR = Path.cwd() / "src"

Файлы оказались задублированы.

Здесь мы сверяем хэши файлов, чтобы выделить дубликаты.

In [ ]:
hash_rows = []

for csv_path in sorted(SRC_DIR.glob("*.csv")):
    hash_rows.append(
        {
            "file_name": csv_path.name,
            "md5": hashlib.md5(csv_path.read_bytes()).hexdigest(),
        }
    )

file_hashes_df = pd.DataFrame(hash_rows).sort_values(["md5", "file_name"]).reset_index(drop=True)
file_hashes_df["duplicate_files_count"] = file_hashes_df.groupby("md5")["file_name"].transform("size")
file_hashes_df["is_duplicate"] = file_hashes_df["duplicate_files_count"].gt(1)

# полный список файлов с отеткой о наличии дубликатов
display(file_hashes_df)

,file_name,md5,duplicate_files_count,is_duplicate
0,user_answers.csv,0b31d71ef836eb434a92529f32d4fb10,2,True
1,wk_users_courses_actions.csv,0b31d71ef836eb434a92529f32d4fb10,2,True
2,user_activity_histories.csv,10924b59e8d371881bf180ae81793ffb,1,False
3,users.csv,1e9774b67af09775e40f0f6a6b8ad8f2,1,False
4,wk_media_view_sessions.csv,9b6757ee4bfc98842fc38266cd9284f6,1,False
5,user_access_histories.csv,a38f01cb6aa085dafef34c8870dacb0c,2,True
6,xp_ratings.csv,a38f01cb6aa085dafef34c8870dacb0c,2,True
7,user_trainings.csv,b85f72846fa2c7049ec6c59be29d079d,1,False
8,award_badges.csv,b984830fac28ddd3e926bc1fa64b7cde,1,False
9,lessons.csv,badd419a9a6b2c9aaec50f25e348c7e6,3,True


## Импорт уникальных файлов

По хэшам остаётся такой набор уникальных CSV. Именно его импортируем дальше.

In [ ]:
unique_files = [
    "award_badges.csv",
    "lessons.csv",
    "user_access_histories.csv",
    "user_activity_histories.csv",
    "user_answers.csv",
    "user_award_badges.csv",
    "user_trainings.csv",
    "users.csv",
    "users_courses.csv",
    "wk_media_view_sessions.csv",
]

unique_files

['award_badges.csv',
 'lessons.csv',
 'user_access_histories.csv',
 'user_activity_histories.csv',
 'user_answers.csv',
 'user_award_badges.csv',
 'user_trainings.csv',
 'users.csv',
 'users_courses.csv',
 'wk_media_view_sessions.csv']

Импортировать данные надо уже с учетом формата данных. 

Сразу распознаем дату/время в формат даты/времени (вместо текстового).

Кроме того, в users.csv и users_courses.csv в числовых параметрах тысячи отделяются запятой.



In [ ]:
award_badges_df = pd.read_csv(SRC_DIR / "award_badges.csv", low_memory=False)

lessons_df = pd.read_csv(SRC_DIR / "lessons.csv", low_memory=False)
lessons_df["wk_attendance_tracking_disabled_at"] = pd.to_datetime(
    lessons_df["wk_attendance_tracking_disabled_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)

user_access_histories_df = pd.read_csv(SRC_DIR / "user_access_histories.csv", low_memory=False)
user_access_histories_df["access_started_at"] = pd.to_datetime(
    user_access_histories_df["access_started_at"],
    format="%d %b, %Y",
    errors="coerce",
)
user_access_histories_df["access_expired_at"] = pd.to_datetime(
    user_access_histories_df["access_expired_at"],
    format="%d %b, %Y",
    errors="coerce",
)

user_activity_histories_df = pd.read_csv(SRC_DIR / "user_activity_histories.csv", low_memory=False)
user_activity_histories_df["created_at"] = pd.to_datetime(
    user_activity_histories_df["created_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)

user_answers_df = pd.read_csv(SRC_DIR / "user_answers.csv", low_memory=False)
user_answers_df["submitted_at"] = pd.to_datetime(
    user_answers_df["submitted_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)

user_award_badges_df = pd.read_csv(SRC_DIR / "user_award_badges.csv", low_memory=False)
user_award_badges_df["created_at"] = pd.to_datetime(
    user_award_badges_df["created_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)

user_trainings_df = pd.read_csv(SRC_DIR / "user_trainings.csv", low_memory=False)
user_trainings_df["started_at"] = pd.to_datetime(
    user_trainings_df["started_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)
user_trainings_df["finished_at"] = pd.to_datetime(
    user_trainings_df["finished_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)
user_trainings_df["mark_saved_at"] = pd.to_datetime(
    user_trainings_df["mark_saved_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)

users_df = pd.read_csv(SRC_DIR / "users.csv", low_memory=False, thousands=",")
users_df["created_at"] = pd.to_datetime(users_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["updated_at"] = pd.to_datetime(users_df["updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["remember_created_at"] = pd.to_datetime(users_df["remember_created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["current_sign_in_at"] = pd.to_datetime(users_df["current_sign_in_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["last_sign_in_at"] = pd.to_datetime(users_df["last_sign_in_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["grade_changed_at"] = pd.to_datetime(users_df["grade_changed_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_df["d_updated_at"] = pd.to_datetime(users_df["d_updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")

users_courses_df = pd.read_csv(SRC_DIR / "users_courses.csv", low_memory=False, thousands=",")
users_courses_df["created_at"] = pd.to_datetime(users_courses_df["created_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["updated_at"] = pd.to_datetime(users_courses_df["updated_at"], format="%d %b, %Y, %H:%M", errors="coerce")
users_courses_df["access_finished_at"] = pd.to_datetime(users_courses_df["access_finished_at"], format="%d %b, %Y", errors="coerce")
users_courses_df["wk_officially_started_at"] = pd.to_datetime(users_courses_df["wk_officially_started_at"], format="%d %b, %Y", errors="coerce")
users_courses_df["wk_course_completed_at"] = pd.to_datetime(users_courses_df["wk_course_completed_at"], format="%d %b, %Y, %H:%M", errors="coerce")

wk_media_view_sessions_df = pd.read_csv(SRC_DIR / "wk_media_view_sessions.csv", low_memory=False)
wk_media_view_sessions_df["reviewed_at"] = pd.to_datetime(
    wk_media_view_sessions_df["reviewed_at"],
    format="%d %b, %Y, %H:%M",
    errors="coerce",
)

dataframes = {
    "award_badges_df": award_badges_df,
    "lessons_df": lessons_df,
    "user_access_histories_df": user_access_histories_df,
    "user_activity_histories_df": user_activity_histories_df,
    "user_answers_df": user_answers_df,
    "user_award_badges_df": user_award_badges_df,
    "user_trainings_df": user_trainings_df,
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
}

list(dataframes.keys())

['award_badges_df',
 'lessons_df',
 'user_access_histories_df',
 'user_activity_histories_df',
 'user_answers_df',
 'user_award_badges_df',
 'user_trainings_df',
 'users_df',
 'users_courses_df',
 'wk_media_view_sessions_df']

In [ ]:
# смотрим размеры датафреймов
all_dataframes = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
    }
)
all_dataframes

,dataframe,rows,cols
0,award_badges_df,6,7
1,lessons_df,3369,12
2,user_access_histories_df,667124,5
3,user_activity_histories_df,1048575,4
4,user_answers_df,1048575,14
5,user_award_badges_df,252843,4
6,user_trainings_df,427628,13
7,users_df,95395,21
8,users_courses_df,290835,14
9,wk_media_view_sessions_df,1048575,7


user_activity_histories_df, user_answers_df, wk_media_view_sessions_df имеют 1048575 строк. Это - максимальное количество строк таблицы excel. Есть подозрение, что файлы могут быть обрезаны.

In [ ]:
for name, df in dataframes.items():
    #дропаем везде явно заданный индекс
    dataframes[name] = dataframes[name].drop(columns='Unnamed: 0') 
    print(name)
    display(df.describe(include="all"))


,Unnamed: 0,name,title,level,quota,special,unlocked_small_image_url
0,0,AwardBadges::OlympiadParticipant,Олимпиадник,1,1,True,https://u.foxford.ngcdn.ru/uploads/inner_file/...
1,1,AwardBadges::Solving,Я решаю,1,5,False,https://u.foxford.ngcdn.ru/uploads/inner_file/...
2,2,AwardBadges::Solving,Я решаю,2,25,False,https://u.foxford.ngcdn.ru/uploads/inner_file/...
3,3,AwardBadges::Solving,Я решаю,3,50,False,https://u.foxford.ngcdn.ru/uploads/inner_file/...
4,4,AwardBadges::Solving,Я решаю,4,100,False,https://u.foxford.ngcdn.ru/uploads/inner_file/...
5,5,AwardBadges::Solving,Я решаю,5,500,False,https://u.foxford.ngcdn.ru/uploads/inner_file/...


award_badges_df


,name,title,level,quota,special,unlocked_small_image_url
count,6,6,6.000000,6.000000,6,6
unique,2,2,NaN,NaN,2,6
top,AwardBadges::Solving,Я решаю,NaN,NaN,False,https://u.foxford.ngcdn.ru/uploads/inner_file/...
freq,5,5,NaN,NaN,5,1
mean,NaN,NaN,2.666667,113.500000,NaN,NaN
std,NaN,NaN,1.632993,192.799118,NaN,NaN
min,NaN,NaN,1.000000,1.000000,NaN,NaN
25%,NaN,NaN,1.250000,10.000000,NaN,NaN
50%,NaN,NaN,2.500000,37.500000,NaN,NaN
75%,NaN,NaN,3.750000,87.500000,NaN,NaN


lessons_df


,course_id,conspect_expected,task_expected,lesson_number,wk_max_points,wk_task_count,wk_survival_training_expected,wk_scratch_playground_enabled,wk_attendance_tracking_enabled,wk_video_duration,wk_attendance_tracking_disabled_at
count,3369,3369,3369,845.000000,2759.000000,2759.000000,3369,3369,3369,1646.000000,2
unique,137,2,2,NaN,NaN,NaN,2,2,2,NaN,NaN
top,"1,032",True,True,NaN,NaN,NaN,False,False,False,NaN,NaN
freq,115,2204,2477,NaN,NaN,NaN,3367,3351,2771,NaN,NaN
mean,NaN,NaN,NaN,25.171598,8.767090,8.815875,NaN,NaN,NaN,4.744228,2026-01-06 04:45:00
min,NaN,NaN,NaN,1.000000,0.000000,0.000000,NaN,NaN,NaN,1.000000,2025-12-23 22:25:00
25%,NaN,NaN,NaN,7.000000,4.000000,4.000000,NaN,NaN,NaN,1.000000,2025-12-30 13:35:00
50%,NaN,NaN,NaN,15.000000,8.000000,7.000000,NaN,NaN,NaN,1.000000,2026-01-06 04:45:00
75%,NaN,NaN,NaN,36.000000,13.000000,13.000000,NaN,NaN,NaN,8.000000,2026-01-12 19:55:00
max,NaN,NaN,NaN,115.000000,24.000000,23.000000,NaN,NaN,NaN,31.000000,2026-01-19 11:05:00


...

,last_explainer_seen_→_course,created_at,updated_at,type,remember_created_at,sign_in_count,current_sign_in_at,last_sign_in_at,grade_id,subscribed,grade_checked,is_teacher,timezone,grade_changed_at,xp,d_wk_school_id,d_wk_municipal_id,d_wk_region_id,d_updated_at,wk_gender
count,8217.000000,95395,95395,95395,83809,95395.000000,95133,95133,95395.000000,95395,95395,95395,95217,1804,95395.00000,64093.000000,64093.000000,64093.000000,95155,15946.000000
unique,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,2,2,1,141,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,User::Pupil,NaN,NaN,NaN,NaN,NaN,True,False,False,Europe/Moscow,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,90647,NaN,NaN,NaN,NaN,NaN,94940,77598,95395,64085,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,6.382743,2025-09-02 10:04:45.591068928,2025-11-25 07:39:32.594789888,NaN,2025-09-19 21:10:17.329403392,18.956077,2025-11-01 03:49:28.553498624,2025-10-23 13:33:19.951436288,3009.521076,NaN,NaN,NaN,NaN,2025-08-30 03:11:38.148558848,249.30380,170094.386407,157136.316478,156203.374097,2025-10-15 15:04:18.307393024,1.456854
min,1.000000,2025-01-31 14:16:00,2025-02-17 07:30:00,NaN,2025-01-31 14:16:00,0.000000,2025-02-17 07:14:00,2025-02-17 06:46:00,3000.000000,NaN,NaN,NaN,NaN,2025-02-04 15:51:00,0.00000,93422.000000,93421.000000,93420.000000,2025-02-04 15:58:00,1.000000
25%,7.000000,2025-06-05 19:58:30,2025-11-18 20:30:30,NaN,2025-06-21 07:21:00,4.000000,2025-09-16 10:39:00,2025-08-31 12:06:00,3010.000000,NaN,NaN,NaN,NaN,2025-06-11 10:44:45,0.00000,144581.000000,144580.000000,144579.000000,2025-10-23 01:40:00,1.000000
50%,7.000000,2025-09-30 11:31:00,2025-12-16 12:16:00,NaN,2025-10-22 03:51:00,9.000000,2025-12-01 19:01:00,2025-11-25 12:30:00,3010.000000,NaN,NaN,NaN,NaN,2025-09-23 08:40:30,249.00000,144581.000000,144580.000000,144579.000000,2025-11-05 13:59:00,1.000000
75%,7.000000,2025-11-12 08:38:00,2025-12-25 01:55:00,NaN,2025-11-19 19:20:00,19.000000,2025-12-18 18:09:00,2025-12-14 19:45:00,3010.000000,NaN,NaN,NaN,NaN,2025-10-28 10:24:15,378.00000,159759.000000,146830.000000,144671.000000,2025-11-17 11:55:30,2.000000
max,7.000000,2026-03-27 16:13:00,2026-03-27 16:47:00,NaN,2026-03-27 16:44:00,1390.000000,2026-03-27 16:47:00,2026-03-27 16:47:00,3012.000000,NaN,NaN,NaN,NaN,2026-03-26 14:58:00,4598.00000,331367.000000,331184.000000,286067.000000,2026-03-27 16:13:00,2.000000


users_courses_df


,user_id,course_id,state,created_at,updated_at,group_template_id,access_finished_at,wk_points,wk_max_points,wk_max_viewable_lessons,wk_max_task_count,wk_officially_started_at,wk_course_completed_at
count,290835.000000,2.908350e+05,290835,290835,290835,2.907680e+05,290741,205925.000000,290710.000000,290710.000000,290710.000000,96865,354
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,287548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,715722.751409,6.916814e+06,NaN,2025-10-02 12:20:35.549848064,2025-10-25 12:07:41.910774272,6.918437e+06,2026-03-25 09:34:34.592850688,52.373126,78.579207,12.288298,82.034202,2025-10-10 10:26:52.297527296,2026-02-11 04:37:16.779661056
min,665740.000000,7.540000e+02,NaN,2025-02-07 11:33:00,2025-02-19 07:00:00,7.650000e+02,2025-02-16 00:00:00,0.000000,0.000000,0.000000,0.000000,2023-06-21 00:00:00,2026-01-12 14:35:00
25%,692031.000000,7.700000e+02,NaN,2025-07-31 00:33:00,2025-09-15 19:34:00,7.810000e+02,2026-03-10 00:00:00,28.000000,4.000000,5.000000,13.000000,2025-09-29 00:00:00,2026-01-24 20:36:30
50%,716070.000000,7.710000e+02,NaN,2025-11-05 12:25:00,2025-11-19 13:03:00,7.820000e+02,2026-05-06 00:00:00,52.300000,63.000000,14.000000,63.000000,2025-11-07 00:00:00,2026-02-08 00:31:00
75%,739958.000000,8.360000e+02,NaN,2025-11-28 06:49:00,2025-12-06 18:04:30,8.470000e+02,2026-05-29 00:00:00,61.330000,79.000000,14.000000,91.000000,2025-11-13 00:00:00,2026-03-01 16:36:30
max,761578.000000,1.700007e+08,NaN,2026-03-26 20:23:00,2026-03-27 01:49:00,1.700007e+08,2026-09-26 00:00:00,492.080000,1230.000000,80.000000,1230.000000,2026-02-23 00:00:00,2026-03-26 17:26:00


wk_media_view_sessions_df


,question_id,reviewed_at,state,count,current_points,recommended_points
count,1.048575e+06,18552,1.048575e+06,1.048575e+06,1.048575e+06,19402.000000
mean,1.108341e+06,2021-10-26 05:27:54.553687040,1.036220e+00,8.227126e+00,1.533974e-02,1.315834
min,1.005165e+06,2021-09-17 17:20:00,1.000000e+00,1.000000e+00,0.000000e+00,0.010000
25%,1.107411e+06,2021-10-02 16:45:00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000
50%,1.108620e+06,2021-10-03 14:23:00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000
75%,1.109265e+06,2021-10-03 17:11:00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000
max,1.140730e+06,2022-10-07 09:16:00,4.000000e+00,1.105500e+04,8.000000e+00,8.000000
std,3.135785e+03,NaN,2.652793e-01,1.187640e+02,1.935579e-01,0.752726


## Базовые проверки качества

In [ ]:
quality_report = pd.DataFrame(
    [
        {
            "dataframe": name,
            "rows": len(df),
            "cols": len(df.columns),
            "duplicate_rows": int(df.duplicated().sum()),
            "all_null_rows": int(df.isna().all(axis=1).sum()),
            "all_null_cols": int(df.isna().all(axis=0).sum()),
            "total_missing": int(df.isna().sum().sum()),
            "columns_with_missing": int((df.isna().sum() > 0).sum())
        }
        for name, df in dataframes.items()
    ]
).sort_values(["total_missing", "rows"], ascending=[False, False])

missing_report = pd.DataFrame(
    [
        {
            "dataframe": name,
            "column": column,
            "missing_count": int(missing_count),
            "missing_pct": round(float(missing_count / len(df) * 100), 2),
        }
        for name, df in dataframes.items()
        for column, missing_count in df.isna().sum().sort_values(ascending=False).head(5).items()
        if missing_count > 0
    ]
).sort_values(["missing_pct", "missing_count"], ascending=[False, False])

display(quality_report)
display(missing_report)

,dataframe,rows,cols,duplicate_rows,all_null_rows,all_null_cols,total_missing,columns_with_missing
9,wk_media_view_sessions_df,1048575,7,0,0,0,2059196,2
4,user_answers_df,1048575,14,0,0,0,1734862,4
8,users_courses_df,290835,14,0,0,0,569897,8
7,users_df,95395,21,0,0,0,366652,11
6,user_trainings_df,427628,13,0,0,0,13268,4
1,lessons_df,3369,12,0,0,0,8834,5
3,user_activity_histories_df,1048575,4,0,0,0,0,0
2,user_access_histories_df,667124,5,0,0,0,0,0
5,user_award_badges_df,252843,4,0,0,0,0,0
0,award_badges_df,6,7,0,0,0,0,0


,dataframe,column,missing_count,missing_pct
0,lessons_df,wk_attendance_tracking_disabled_at,3367,99.94
18,users_courses_df,wk_course_completed_at,290481,99.88
23,wk_media_view_sessions_df,reviewed_at,1030023,98.23
24,wk_media_view_sessions_df,recommended_points,1029173,98.15
13,users_df,grade_changed_at,93591,98.11
14,users_df,last_explainer_seen_→_course,87178,91.39
15,users_df,wk_gender,79449,83.28
5,user_answers_df,skipped,854881,81.53
6,user_answers_df,wk_partial_answer,854881,81.53
1,lessons_df,lesson_number,2524,74.92
